Tạo Spark Session:

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
    .appName("FeatureEngineering")\
    .config("spark.driver.memory", "4g")\
    .config("spark.sql.shuffle.partitions", "8")\
    .getOrCreate()

Load datasets:


In [2]:
df_orders = spark.read.csv(path= "/home/jovyan/data/orders.csv", header= True, inferSchema= True)
# df_prior = spark.read.csv(path="/home/jovyan/data/order_products__prior.csv", header= True, inferSchema= True)
df_prior = spark.read.csv("/home/jovyan/data/order_products__prior.csv", header=True, inferSchema=True).sample(fraction=0.1, seed=42)
df_train = spark.read.csv(path="/home/jovyan/data/order_products__train.csv", header= True, inferSchema= True)
df_products = spark.read.csv(path="/home/jovyan/data/products.csv", header= True, inferSchema= True)
df_aisles = spark.read.csv(path="/home/jovyan/data/aisles.csv", header= True, inferSchema= True)
df_departments = spark.read.csv(path="/home/jovyan/data/departments.csv", header= True, inferSchema= True)

df_orders.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- eval_set: string (nullable = true)
 |-- order_number: integer (nullable = true)
 |-- order_dow: integer (nullable = true)
 |-- order_hour_of_day: integer (nullable = true)
 |-- days_since_prior_order: double (nullable = true)



Xem các cột và sửa giống như phần exploration:

In [3]:
print(f'order: {df_orders.printSchema()}')
print(f'aisles: {df_aisles.printSchema()}')
print(f'department: {df_departments.printSchema()}')
print(f'product: {df_products.printSchema()}')
print(f'train: {df_train.printSchema()}')
print(f'prior: {df_prior.printSchema()}')

root
 |-- order_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- eval_set: string (nullable = true)
 |-- order_number: integer (nullable = true)
 |-- order_dow: integer (nullable = true)
 |-- order_hour_of_day: integer (nullable = true)
 |-- days_since_prior_order: double (nullable = true)

order: None
root
 |-- aisle_id: integer (nullable = true)
 |-- aisle: string (nullable = true)

aisles: None
root
 |-- department_id: integer (nullable = true)
 |-- department: string (nullable = true)

department: None
root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- aisle_id: string (nullable = true)
 |-- department_id: string (nullable = true)

product: None
root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- add_to_cart_order: integer (nullable = true)
 |-- reordered: integer (nullable = true)

train: None
root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (null

In [4]:
from pyspark.sql.functions import col
df_products = df_products.withColumn('department_id', col('department_id').cast('integer'))
df_products = df_products.withColumn('aisle_id', col('aisle_id').cast('integer'))

In [5]:
# Quick schema reference
print("=== ORDERS ===");        df_orders.printSchema()
print("=== PRIOR ===");         df_prior.printSchema()
print("=== TRAIN ===");         df_train.printSchema()
print("=== PRODUCTS ===");      df_products.printSchema()

=== ORDERS ===
root
 |-- order_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- eval_set: string (nullable = true)
 |-- order_number: integer (nullable = true)
 |-- order_dow: integer (nullable = true)
 |-- order_hour_of_day: integer (nullable = true)
 |-- days_since_prior_order: double (nullable = true)

=== PRIOR ===
root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- add_to_cart_order: integer (nullable = true)
 |-- reordered: integer (nullable = true)

=== TRAIN ===
root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- add_to_cart_order: integer (nullable = true)
 |-- reordered: integer (nullable = true)

=== PRODUCTS ===
root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- aisle_id: integer (nullable = true)
 |-- department_id: integer (nullable = true)



Đặc trưng của user:

In [6]:
from pyspark.sql.functions import count, mean, countDistinct, max

df_orders_prior = df_orders.join(df_prior, on= 'order_id', how= 'left').cache()

user_features = df_orders.groupBy('user_id').agg(
    count('order_id').alias('total_orders'),
    mean('days_since_prior_order'). alias('avg_days_between_orders')

)

basket_per_order = df_orders_prior.groupBy('user_id', 'order_id').agg(
    count('product_id').alias('basket_size')
)

user_basket = basket_per_order.groupBy('user_id').agg(
    mean('basket_size').alias('avg_basket_size')
)
user_reorder = df_orders_prior.groupBy('user_id').agg(
    mean('reordered').alias('overall_reorder_rate')
)

user_features = user_features.join(user_basket, on= 'user_id', how= 'left').join(user_reorder, on= 'user_id', how= 'left')

Đặc trung cho products:

In [7]:
product_features = df_prior.groupBy('product_id').agg(
    count('order_id').alias('product_total_orders')
)

product_reorder_rate_users = df_orders_prior.groupBy('product_id').agg(
    mean('reordered').alias('reorder_rate'),
    countDistinct('user_id').alias('unique_user')
)

product_features = product_features.join(product_reorder_rate_users, on= 'product_id', how= 'left')

Đặc trưng cho user-product:

In [8]:
up_features = df_orders_prior.groupBy('user_id', 'product_id').agg(
    count('order_id').alias('up_times_bought'),
    mean('reordered').alias('up_reorder_rate'),
    mean('add_to_cart_order').alias('up_avg_cart_pos'),
    max('order_number').alias('up_last_order')
)

In [9]:
df_unique_prior = df_orders_prior.select('user_id', 'product_id')
df_unique_prior = df_unique_prior.dropDuplicates()

df_orders_train = df_train.join(df_orders, on= 'order_id', how= 'left')
df_unique_train = df_orders_train.select('user_id', 'product_id')
df_unique_train = df_unique_train.dropDuplicates()


from pyspark.sql.functions import when, lit
df_unique_train = df_unique_train.withColumn('label', lit(1))
df_unique = df_unique_prior.join(df_unique_train, on= ['product_id', 'user_id'], how='left')
df_unique = df_unique.withColumn('label', when(col('label').isNull(), 0).otherwise(1))

In [10]:
df_unique = df_unique.join(user_features, on='user_id', how='left') \
    .join(product_features, on='product_id', how='left') \
    .join(up_features, on=['product_id', 'user_id'], how='left')


df_unique.show() 

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving
ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 516, in send_com

Py4JError: An error occurred while calling o217.showString

In [ ]:
from pyspark.sql.functions import count, mean

df_orders_prior = df_orders.join(df_prior, on='order_id', how='left')

user_features = df_orders.groupBy('user_id').agg(
    count('order_id').alias('total_orders'),
    mean('days_since_prior_order').alias('avg_days_between_orders')
)

basket_per_order = df_orders_prior.groupBy('user_id', 'order_id').agg(
    count('product_id').alias('basket_size')
)

user_basket = basket_per_order.groupBy('user_id').agg(
    mean('basket_size').alias('avg_basket_size')
)

user_reorder = df_orders_prior.groupBy('user_id').agg(
    mean('reordered').alias('overall_reorder_rate')
)

user_features = user_features \
    .join(user_basket, on='user_id', how='left') \
    .join(user_reorder, on='user_id', how='left')

user_features.write.parquet('/home/jovyan/data/features/user_features')



In [5]:
from pyspark.sql.functions import countDistinct

product_features = df_orders_prior.groupBy('product_id').agg(
    count('order_id').alias('product_total_orders'),
    mean('reordered').alias('product_reorder_rate'),
    countDistinct('user_id').alias('product_unique_users')
)

product_features.write.parquet('/home/jovyan/data/features/product_features')

In [7]:
from pyspark.sql.functions import max
up_features = df_orders_prior.groupBy('user_id', 'product_id').agg(
    count('order_id').alias('up_times_bought'),
    mean('reordered').alias('up_reorder_rate'),
    mean('add_to_cart_order').alias('up_avg_cart_pos'),
    max('order_number').alias('up_last_order')
)

up_features.write.parquet('/home/jovyan/data/features/up_features')

In [8]:
from pyspark.sql.functions import when, lit, col

# Base table
df_unique_prior = df_orders_prior.select('user_id', 'product_id').dropDuplicates()

# Train labels
df_orders_train = df_train.join(df_orders, on='order_id', how='left')
df_unique_train = df_orders_train.select('user_id', 'product_id').dropDuplicates()
df_unique_train = df_unique_train.withColumn('label', lit(1))

# Join to create labels
df_base = df_unique_prior.join(df_unique_train, on=['user_id', 'product_id'], how='left')
df_base = df_base.withColumn('label', when(col('label').isNull(), 0).otherwise(1))

# Load saved features
user_features = spark.read.parquet('/home/jovyan/data/features/user_features')
product_features = spark.read.parquet('/home/jovyan/data/features/product_features')
up_features = spark.read.parquet('/home/jovyan/data/features/up_features')

# Join all features
df_final = df_base \
    .join(user_features, on='user_id', how='left') \
    .join(product_features, on='product_id', how='left') \
    .join(up_features, on=['user_id', 'product_id'], how='left')

df_final.write.parquet('/home/jovyan/data/features/final_dataset')

In [9]:
df_final = spark.read.parquet('/home/jovyan/data/features/final_dataset')
df_final.printSchema()
df_final.count()

root
 |-- user_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- label: integer (nullable = true)
 |-- total_orders: long (nullable = true)
 |-- avg_days_between_orders: double (nullable = true)
 |-- avg_basket_size: double (nullable = true)
 |-- overall_reorder_rate: double (nullable = true)
 |-- product_total_orders: long (nullable = true)
 |-- product_reorder_rate: double (nullable = true)
 |-- product_unique_users: long (nullable = true)
 |-- up_times_bought: long (nullable = true)
 |-- up_reorder_rate: double (nullable = true)
 |-- up_avg_cart_pos: double (nullable = true)
 |-- up_last_order: integer (nullable = true)



2779603

In [10]:
df_final.groupBy('label').count().show()

+-----+-------+
|label|  count|
+-----+-------+
|    1| 272603|
|    0|2507000|
+-----+-------+

